# Snowflake × AI によるクオンツ・ポートフォリオ最適化

**Quantitative Portfolio Optimization with Snowflake AI Data Cloud**

> 所要時間: 約3時間30分 ／ 環境: Snowflake トライアルアカウント（GPU不要）

## ハンズオン全体の流れ

| Block | テーマ | 主要機能 |
|-------|--------|---------|
| **0** | 環境セットアップ・コストコントロール | Warehouse, Resource Monitor |
| **1A** | 有報 PDF → RAG 検索パイプライン | AI_PARSE_DOCUMENT, Cortex Search |
| **1B** | 決算トランスクリプト × センチメント | AI_COMPLETE, AI_FILTER |
| **2** | 5ファクターモデル + IC 統計検定 | Snowflake Notebook, LightGBM, ML Registry |
| **3** | CVaR ポートフォリオ最適化 SP | Python Stored Procedure, scipy |
| **4** | Cortex Code × AI エージェント構築 | Cortex Code, Semantic View, Cortex Agent |
| **4.5** | Agent Skills 登録 | SKILL.md, ALTER AGENT |
| **5** | Snowflake Intelligence デモ | Snowflake Intelligence |

> **Cortex Code を積極的に活用してください。**  
> 画面右下の ✦ アイコンをクリックして起動。`@テーブル名` でコンテキスト追加、`Fix` ボタンでエラー自動修正、`Plan` モードで複雑な多段タスクを設計できます。


---

## クオンツ分析におけるSnowflakeの戦略的価値

### 従来のクオンツワークフローが抱える4つの構造的課題

クオンツチームが日々直面するワークフローを分解すると、以下の4つの非効率が見えてきます。

```
[従来のワークフロー]

市場データDB ──EXTRACT──▶ ローカル環境 ──TRANSFORM──▶ 分析スクリプト
有報PDF      ──手動OCR──▶ テキストDB   ──手動クレンジング──▶ NLP処理
                                                              │
                                                        コード変更のたびに
                                                        データ再取得・再処理
                                                              │
                                                        ▼
                                                   モデル登録はExcelや
                                                   ローカルpickleファイル
```

| 課題 | 症状 | 影響 |
|------|------|------|
| **① データ移動の繰り返し** | 分析のたびにDBからデータをExport | セキュリティリスク・時間コスト |
| **② 非構造化データの属人化** | 有報の読み込み・要約が担当者依存 | カバレッジ不足・情報の非対称 |
| **③ モデルの再現性不足** | pickleファイル管理・依存関係の齟齬 | バージョン管理困難・本番化が難しい |
| **④ システム間の統合コスト** | ファクターDB・最適化ツール・バックテストが別々 | 戦略開発のリードタイム増大 |

---

### Snowflakeがこれを解決する仕組み：**Compute to Data**

従来の「データをツールに持ってくる」発想を逆転し、**データがある場所に計算を持ち込む**アーキテクチャです。

```
[Snowflakeのアーキテクチャ]

Snowflake Data Cloud
┌────────────────────────────────────────────────────────────────┐
│                                                                │
│  Marketplace（株価・SEC・決算書）                               │
│      │ ゼロコピー共有（データは移動しない）                      │
│      ▼                                                        │
│  Snowflake Notebook ─── Python/SQL が Data の隣で実行          │
│      │                                                        │
│  AI Functions（AI_COMPLETE / AI_PARSE_DOCUMENT）              │
│      │ LLMがSnowflake内で実行 → 有報テキストが外に出ない         │
│      ▼                                                        │
│  ML Registry ─── モデルをSnowflakeオブジェクトとして管理        │
│      │                                                        │
│  Cortex Agent ─── 全ツールを自然言語で統合オーケストレーション   │
│                                                                │
└────────────────────────────────────────────────────────────────┘
                    ↑ RBAC・監査・暗号化がインフラとして標準装備
```

---

### クオンツチームが得る4つのメリット

#### 1. ゼロコピー・データインプレース処理
Snowflake Marketplaceから株価・SEC財務データを「取得」しても、**データは提供者側に存在したまま**です。  
クエリのたびに手元にExportする必要がなく、常に最新データへのアクセスが保証されます。

#### 2. 有報・決算資料の大量処理
`AI_COMPLETE()` と `AI_PARSE_DOCUMENT()` により、  
これまで担当アナリストが手動で行っていた**有報の読み込み・要約・センチメント判定**を  
全上場企業を対象にSQLの1クエリで実行できます。

```sql
-- 例: 全銘柄の決算トランスクリプトをセンチメント分析
SELECT TICKER, AI_COMPLETE('claude-4-sonnet', 
    '{"score":..., "risks":...}をJSONで返して: ' || TRANSCRIPT_TEXT)
FROM EARNINGS_TRANSCRIPTS;
-- → 従来: アナリスト1人・1社・数時間
-- → Snowflake: 全銘柄・数分
```

#### 3. 機械学習モデルの再現性と組織共有
`ML Registry` により、LightGBMなどのモデルを**バージョン管理・メタデータ付き**でSnowflake内に保存できます。  
モデルは `Stored Procedure` として公開でき、SQL1行で誰でも同一モデルを呼び出せます。

```sql
-- モデルをSQL関数として呼び出す
SELECT FACTOR_RETURN_PREDICTOR!PREDICT(Z_MOMENTUM, Z_VALUE, ...)
FROM FACTOR_SCORES_LATEST;
```

#### 4. 分析から意思決定までの統合インターフェース
Cortex Agent + Snowflake Intelligence により、  
**「スクリーニング → ファンダメンタル確認 → ポートフォリオ最適化」を1つの自然言語会話で完結**できます。

---

### 本ハンズオンで構築するもの

| コンポーネント | 技術 | クオンツ的価値 |
|-------------|------|-------------|
| 有報RAGパイプライン | AI_PARSE_DOCUMENT + Cortex Search | 日本語・英語文書の大量処理 |
| センチメントファクター | AI_COMPLETE × 決算トランスクリプト | テキスト → 定量シグナル |
| 5ファクターモデル | LightGBM + ML Registry | 再現性・組織共有 |
| CVaR最適化エンジン | scipy + Python SP | セクター制約付き機関品質最適化 |
| Quant PM Agent | Cortex Agent × 6ツール | 意思決定の自動化・加速 |

> **所要時間の目安: 約3時間30分**  
> セルは上から順に実行してください。  
> Cortex Code（右下 ✦ アイコン）を積極的に活用し、エラー修正・コード理解に使ってください。


---
## Block 0: 環境セットアップ・コストコントロール（20分）

### データベース・スキーマ・ウェアハウスの設計方針

| オブジェクト | 名前 | 用途 |
|------------|------|------|
| Database | `QUANT_HOL_DB` | ハンズオン全体のデータ格納場所 |
| Schema: QUANT | 株価・ファクター・有報データ | 分析・モデリング層 |
| Schema: PM | ポートフォリオ最適化 | PM層 |
| Schema: AI | Semantic View・エージェント | AI層 |
| Warehouse | `QUANT_HOL_WH` (Medium) | クエリ・SP 実行 |

### コストコントロールの重要性

本番環境では Strategy 別・担当者別に Resource Monitor を設定することで、  
ファクターリサーチのコスト配賦が可能になります。  
`SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY` でリアルタイム監視ができます。


In [ ]:
-- データベース・スキーマ作成
CREATE DATABASE IF NOT EXISTS QUANT_HOL_DB;
CREATE SCHEMA  IF NOT EXISTS QUANT_HOL_DB.QUANT;
CREATE SCHEMA  IF NOT EXISTS QUANT_HOL_DB.PM;
CREATE SCHEMA  IF NOT EXISTS QUANT_HOL_DB.AI;

-- ウェアハウス作成（Medium: 4クレジット/時間）
CREATE WAREHOUSE IF NOT EXISTS QUANT_HOL_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND   = 60
    AUTO_RESUME    = TRUE
    COMMENT        = 'クオンツ・ポートフォリオ最適化ハンズオン用';

USE DATABASE  QUANT_HOL_DB;
USE WAREHOUSE QUANT_HOL_WH;


In [ ]:
-- Resource Monitor: ハンズオン全体のクレジット上限
-- (1人50クレジット × 想定参加者数で設定)
CREATE OR REPLACE RESOURCE MONITOR QUANT_HOL_MONITOR
    WITH CREDIT_QUOTA = 300
    FREQUENCY        = MONTHLY
    START_TIMESTAMP  = IMMEDIATELY
    TRIGGERS
        ON 50  PERCENT DO NOTIFY
        ON 80  PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND;

ALTER WAREHOUSE QUANT_HOL_WH
    SET RESOURCE_MONITOR = QUANT_HOL_MONITOR;


In [ ]:
-- 本日のクレジット消費確認（随時実行可）
SELECT
    WAREHOUSE_NAME,
    SUM(CREDITS_USED)         AS TOTAL_CREDITS,
    SUM(CREDITS_USED_COMPUTE) AS COMPUTE_CREDITS,
    MAX(END_TIME)             AS LAST_QUERY
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE START_TIME >= CURRENT_DATE()
GROUP BY 1
ORDER BY 2 DESC;


In [ ]:
-- Snowflake Public Data (Free) のテーブル確認
-- ※ 事前に Marketplace から "Snowflake Public Data (Free)" を取得しておくこと

-- 株価データ確認
SELECT TICKER, COUNT(*) AS DATA_POINTS, MIN(DATE) AS FROM_DATE, MAX(DATE) AS TO_DATE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
WHERE VARIABLE = 'post-market_close_adjusted'
  AND TICKER IN ('AAPL','MSFT','NVDA','JPM','JNJ')
GROUP BY 1 ORDER BY 1;


In [ ]:
-- 決算トランスクリプト確認
SELECT TICKERS, EVENT_TIMESTAMP::DATE AS DATE, EVENT_TYPE,
       LEFT(TRANSCRIPT_TEXT::VARCHAR, 200) AS PREVIEW
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.EARNINGS_TRANSCRIPTS
WHERE ARRAY_CONTAINS('AAPL'::VARIANT, TICKERS)
  AND EVENT_TYPE ILIKE '%earnings%'
ORDER BY EVENT_TIMESTAMP DESC
LIMIT 3;


---
## Block 1A: PDF処理パイプライン — AI_PARSE_DOCUMENT → Chunking → Cortex Search（25分）

### データ戦略の整理

本ハンズオンでは非構造化データを以下の2系統で処理します:

| 系統 | データソース | 対象 | 取得方法 |
|------|------------|------|---------|
| **系統①** | EARNINGS_TRANSCRIPTS (Marketplace) | 米国株 決算説明会 | Block 1B で処理 |
| **系統②** | Toyota 半期報告書 PDF (Git repo) | 日本語 IR 資料 | **本 Block 1A で処理** |

> **PDF は Toyota のみで十分な理由**  
> NVDA/AAPL/JPM 等の財務データは `SEC_REPORT_ATTRIBUTES`、  
> 決算トランスクリプトは `EARNINGS_TRANSCRIPTS` として Marketplace で完全にカバー。  
> PDF 処理のデモには**日本語文書**を使うことで  
> 「英語・日本語いずれも同じパイプラインが機能する」ことが示せます。

### 構築するパイプライン

```
toyota_semiannual_2025_09.pdf  (setup.sql Section 7 で自動コピー済み)
    ↓ AI_PARSE_DOCUMENT('text')
    OCR + 構造保持（表・箇条書きも Markdown 形式で抽出）
    ↓ SPLIT_TEXT_RECURSIVE_CHARACTER(1000文字, overlap=200)
    チャンク分割（文脈オーバーラップで境界での情報欠損を防止）
    ↓ CREATE CORTEX SEARCH SERVICE
    ANNUAL_REPORT_SEARCH  ← セマンティック検索エンジン完成
```

### PDF の導入確認

> `setup.sql` の Section 7 を実行済みであれば、  
> トヨタ半期報告書は既に `ANNUAL_REPORTS_STAGE` にコピーされています。  
> 次のセルで確認してください。  
> まだの場合は手動で `docs/toyota_semiannual_2025_09.pdf` をアップロードしてください。


In [ ]:
-- ステージ作成（setup.sql 未実行の場合のみ）
CREATE STAGE IF NOT EXISTS QUANT_HOL_DB.QUANT.ANNUAL_REPORTS_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT   = '半期報告書 PDF 格納場所';


In [ ]:
-- ステージ内ファイル確認（toyota_semiannual_2025_09.pdf が表示されること）
LS @QUANT_HOL_DB.QUANT.ANNUAL_REPORTS_STAGE;


In [ ]:
-- AI_PARSE_DOCUMENT でトヨタ半期報告書を解析
-- ※ 処理に 30秒〜2分かかる場合があります（ページ数に依存）
CREATE OR REPLACE TABLE QUANT_HOL_DB.QUANT.PARSED_ANNUAL_REPORTS AS
SELECT
    RELATIVE_PATH,
    '7203'               AS TICKER,
    'トヨタ自動車'        AS COMPANY_NAME,
    '半期報告書 FY2026Q2' AS REPORT_TYPE,
    AI_PARSE_DOCUMENT(
        TO_FILE('@QUANT_HOL_DB.QUANT.ANNUAL_REPORTS_STAGE', RELATIVE_PATH),
        {'mode': 'LAYOUT'}  -- LAYOUT: 表・見出し・構造を Markdown 形式で保持（財務文書に最適）
    ):content::VARCHAR   AS FULL_TEXT,
    CURRENT_DATE()       AS PARSED_AT
FROM DIRECTORY(@QUANT_HOL_DB.QUANT.ANNUAL_REPORTS_STAGE)
WHERE RELATIVE_PATH ILIKE '%.pdf';

-- 確認: 文字数とプレビュー
SELECT TICKER, COMPANY_NAME, REPORT_TYPE,
       LENGTH(FULL_TEXT) AS TEXT_LEN,
       LEFT(FULL_TEXT, 500) AS PREVIEW
FROM QUANT_HOL_DB.QUANT.PARSED_ANNUAL_REPORTS;


In [ ]:
-- SPLIT_TEXT_RECURSIVE_CHARACTER でチャンク分割
-- 1000文字チャンク × 200文字オーバーラップ（文脈保持）
CREATE OR REPLACE TABLE QUANT_HOL_DB.QUANT.ANNUAL_REPORT_CHUNKS AS
SELECT
    p.TICKER,
    p.COMPANY_NAME,
    p.REPORT_TYPE,
    p.RELATIVE_PATH,
    c.INDEX            AS CHUNK_INDEX,
    c.VALUE::VARCHAR   AS CHUNK_TEXT,
    LENGTH(c.VALUE::VARCHAR) AS CHUNK_LENGTH
FROM QUANT_HOL_DB.QUANT.PARSED_ANNUAL_REPORTS p,
    LATERAL FLATTEN(
        SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
            p.FULL_TEXT,
            'none',
            1000,
            200
        )
    ) c;

-- チャンク統計確認
SELECT TICKER, COMPANY_NAME, REPORT_TYPE,
       COUNT(*) AS CHUNK_COUNT,
       AVG(CHUNK_LENGTH)::INT AS AVG_LEN,
       MIN(CHUNK_LENGTH) AS MIN_LEN,
       MAX(CHUNK_LENGTH) AS MAX_LEN
FROM QUANT_HOL_DB.QUANT.ANNUAL_REPORT_CHUNKS
GROUP BY 1, 2, 3;


In [ ]:
-- Cortex Search Service 作成（インデックス構築に 2〜5分かかります）
CREATE OR REPLACE CORTEX SEARCH SERVICE QUANT_HOL_DB.QUANT.ANNUAL_REPORT_SEARCH
    ON CHUNK_TEXT
    ATTRIBUTES TICKER, COMPANY_NAME, REPORT_TYPE, CHUNK_INDEX
    WAREHOUSE = QUANT_HOL_WH
    TARGET_LAG = '1 day'
AS (
    SELECT TICKER, COMPANY_NAME, REPORT_TYPE, CHUNK_INDEX, CHUNK_TEXT
    FROM QUANT_HOL_DB.QUANT.ANNUAL_REPORT_CHUNKS
);


In [ ]:
-- 有報 Cortex Search のテスト検索（日本語クエリでも動作することを確認）
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'QUANT_HOL_DB.QUANT.ANNUAL_REPORT_SEARCH',
    '{
        "query": "リスク要因 事業環境 市場競争",
        "columns": ["TICKER","COMPANY_NAME","REPORT_TYPE","CHUNK_TEXT"],
        "limit": 3
    }'
) AS SEARCH_RESULT;


---
## Block 1B: 決算トランスクリプト × AI Functions によるセンチメント分析（30分）

### Cortex AI Functions の役割

| 関数 | 役割 |
|------|------|
| `AI_COMPLETE()` | LLM による自然言語タスク（センチメント分析・要約・抽出） |
| `AI_FILTER()` | 条件による自然言語フィルタリング（スクリーニング） |

### センチメント分析の設計指針

本ハンズオンでは決算説明会トランスクリプトから以下を抽出します:
- **sentiment_score (1-10)**: 経営陣の発言トーン → 非財務ファクターとして利用
- **growth_outlook**: 成長見通し (positive/neutral/negative)
- **management_tone**: 経営陣の姿勢 (confident/cautious/mixed)
- **key_risks**: 主要リスク要因 → 有報 RAG 検索のクエリヒントとして活用

> **⚠️ AI Functions のクレジット消費について**  
> トライアルアカウントでは約10クレジット/日の上限があります。  
> 5銘柄のみ処理するため通常は範囲内ですが、超過時は Snowflake に支払い情報を登録してください。


In [ ]:
-- 対象銘柄の直近決算トランスクリプトを確認
SELECT
    TICKERS[0]::VARCHAR     AS TICKER,
    EVENT_TIMESTAMP::DATE   AS EVENT_DATE,
    EVENT_TYPE,
    LEFT(TRANSCRIPT_TEXT::VARCHAR, 300) AS PREVIEW
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.EARNINGS_TRANSCRIPTS
WHERE (
    ARRAY_CONTAINS('AAPL'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('MSFT'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('NVDA'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('JPM'::VARIANT,  TICKERS) OR
    ARRAY_CONTAINS('JNJ'::VARIANT,  TICKERS)
)
AND EVENT_TYPE ILIKE '%earnings%'
QUALIFY ROW_NUMBER() OVER (PARTITION BY TICKERS[0] ORDER BY EVENT_TIMESTAMP DESC) = 1
ORDER BY TICKER;


In [ ]:
-- AI_COMPLETE でセンチメント構造化テーブルを作成
CREATE OR REPLACE TABLE QUANT_HOL_DB.QUANT.EARNINGS_SENTIMENT AS
SELECT
    TICKERS[0]::VARCHAR     AS TICKER,
    EVENT_TIMESTAMP::DATE   AS EVENT_DATE,
    EVENT_TYPE,
    TRY_PARSE_JSON(
        AI_COMPLETE(
            'claude-4-sonnet',
            CONCAT(
                'Analyze this earnings call transcript. Respond in pure JSON only:
',
                '{"sentiment_score":<1-10>,"growth_outlook":"<positive|neutral|negative>",',
                '"key_positives":["<p1>","<p2>"],"key_risks":["<r1>","<r2>"],',
                '"management_tone":"<confident|cautious|mixed>",',
                '"investment_signal":"<strong_buy|buy|hold|sell|strong_sell>"}

',
                'Transcript:
', LEFT(TRANSCRIPT_TEXT::VARCHAR, 4000)
            )
        )
    ) AS SENTIMENT_JSON
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.EARNINGS_TRANSCRIPTS
WHERE (
    ARRAY_CONTAINS('AAPL'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('MSFT'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('NVDA'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('JPM'::VARIANT,  TICKERS) OR
    ARRAY_CONTAINS('JNJ'::VARIANT,  TICKERS)
)
AND EVENT_TYPE ILIKE '%earnings%'
QUALIFY ROW_NUMBER() OVER (PARTITION BY TICKERS[0] ORDER BY EVENT_TIMESTAMP DESC) = 1;


In [ ]:
-- センチメント分析結果の確認
SELECT
    TICKER, EVENT_DATE,
    SENTIMENT_JSON:sentiment_score::INTEGER     AS SENTIMENT_SCORE,
    SENTIMENT_JSON:growth_outlook::VARCHAR      AS GROWTH_OUTLOOK,
    SENTIMENT_JSON:management_tone::VARCHAR     AS MGMT_TONE,
    SENTIMENT_JSON:investment_signal::VARCHAR   AS SIGNAL,
    SENTIMENT_JSON:key_risks                    AS KEY_RISKS
FROM QUANT_HOL_DB.QUANT.EARNINGS_SENTIMENT
ORDER BY SENTIMENT_SCORE DESC;


In [ ]:
-- AI_FILTER: 地政学・中国リスクに言及する銘柄をスクリーニング
SELECT TICKERS[0]::VARCHAR AS TICKER, EVENT_TIMESTAMP::DATE AS DATE
FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.EARNINGS_TRANSCRIPTS
WHERE (
    ARRAY_CONTAINS('AAPL'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('MSFT'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('NVDA'::VARIANT, TICKERS) OR
    ARRAY_CONTAINS('JPM'::VARIANT,  TICKERS)
)
AND AI_FILTER(
    LEFT(TRANSCRIPT_TEXT::VARCHAR, 4000),
    'Does this transcript mention significant geopolitical or China market risk as a major concern?'
)
QUALIFY ROW_NUMBER() OVER (PARTITION BY TICKERS[0] ORDER BY EVENT_TIMESTAMP DESC) = 1;


---
## Block 2: 5ファクターモデル構築と統計的検証（45分）

### なぜ「Snowflake Notebook」でファクターモデルを構築するのか

従来のクオンツワークフローでは、ファクター計算はローカルのPythonスクリプトやJupyter上で行い、  
結果をCSVやDBに書き戻すという**データの往復**が発生していました。  
Snowflake Notebook では**Python がデータの隣で実行**されるため、このオーバーヘッドが消えます。

```
[従来]                              [Snowflake Notebook]
DB ──export──▶ pandas ──calc──▶ DB    DB ◀──▶ pandas (同一環境)
      ↑時間・セキュリティリスク          ↑ゼロコピー・RBAC管理
```

加えて、Block 1B で算出した**AIセンチメントスコアをそのままファクターとして利用**できます。  
これが「非構造化 × 構造化の統合」の核心です。

---

### ファクター投資の理論的背景

本ブロックでは学術的に確立された5つのリスクファクターを実装します。

| ファクター | 理論 | 計算定義 | 実証的根拠 |
|-----------|------|---------|----------|
| **Momentum** | Jegadeesh & Titman (1993) | `ret_12m - ret_1m` | 過去勝者が将来も勝者（短期反転回避のため近月を除く） |
| **Value** | Fama & French (1992) | Revenue Growth（SEC） | 割安銘柄のリスクプレミアム |
| **Quality** | Asness et al. (2019) | `ROE - Leverage × 0.3` | 高収益・低レバレッジ銘柄のプレミアム |
| **Low Volatility** | Frazzini & Pedersen (2014) | `-σ_252` | BAB (Betting Against Beta)：低ボラ銘柄の超過収益 |
| **Sentiment** | Tetlock (2007) | AI_COMPLETE スコア | テキストネガティブ度がリターンを予測 |

### モメンタムファクターの設計詳細

```python
# Jegadeesh & Titman (1993) の定義
# 近月リターン（ret_1m）を除くことで短期反転（mean reversion）を回避
momentum = ret_12m - ret_1m
```

**なぜ1ヶ月を除くのか?**  
株価には短期（1ヶ月以内）の平均回帰傾向があります（DeBondt & Thaler 1985）。  
直近1ヶ月を除いた12-1Mモメンタムにより、この反転バイアスを除去します。

### クロスセクショナルZスコア化の重要性

```python
# 各日付での銘柄間標準化（横断面での比較を可能にする）
z_score = (factor_raw - factor_raw.mean()) / factor_raw.std()
```

**なぜZスコア化が必要か?**  
- ROE（0〜100%レンジ）とモメンタム（-1〜+1レンジ）は単位が異なります
- クロスセクショナルZスコアにより、すべてのファクターを**同一スケールで合成**できます
- これにより等ウェイト合成が「事実上のリスクウェイト」として機能します

---

### IC（情報係数）と統計的有意性の評価基準

**IC（Information Coefficient）** = ファクタースコアと翌月リターンのSpearmanランク相関係数

$$\text{IC}_t = \text{Spearman}\rho(\text{FactorScore}_t, \text{ReturnLead}_{t+21d})$$

| 指標 | 計算式 | 実用水準 | 解釈 |
|------|--------|---------|------|
| **Mean IC** | `IC_monthly.mean()` | > 0.05 | ファクターの平均予測力 |
| **ICIR** | `IC_mean / IC_std` | > 0.5 | シャープレシオのIC版：安定性の指標 |
| **t-統計量** | `IC_mean / (IC_std/√N)` | > 2.0 (5%有意) | ICがゼロと統計的に異なるか |
| **% IC > 0** | `(IC > 0).mean()` | > 55% | ICがプラスになる月の割合 |

> **クオンツの実務基準**: Mean IC > 0.05 かつ t-stat > 2.0 が最低ライン。  
> ICIR > 0.5 であれば機関投資家への採用検討水準。

### LightGBMを選択する理由

| モデル | 特徴 | ファクターモデルへの適合 |
|--------|------|----------------------|
| Linear Regression | 解釈しやすい | 非線形交互作用を捉えられない |
| **LightGBM** | **勾配ブースティング・非線形** | **ファクター間の交互作用を自動学習** |
| Neural Network | 高い表現力 | サンプル数に対して過学習リスク |

ファクター間には非線形の交互作用があります  
（例: 「モメンタムが高く、かつクオリティも高い銘柄」は線形の和では捉えられない）。  
LightGBMはこの構造を効率的に学習します。

### Snowflake ML Registryの価値

モデルを `ML Registry` に登録することで以下が実現します:

```sql
-- 登録されたモデルを SQL関数として呼び出す
SELECT ticker,
       FACTOR_RETURN_PREDICTOR!PREDICT(object_construct(
           'Z_MOMENTUM', z_momentum, 'Z_VALUE', z_value, ...
       )) AS predicted_return
FROM FACTOR_SCORES_LATEST;
```

| 機能 | 価値 |
|------|------|
| **バージョン管理** | V1, V2... と複数バージョンを保持 |
| **メタデータ保存** | IC, p-value, シャープ比を登録時に記録 |
| **SQL関数化** | アナリストがPythonを知らなくてもモデルを利用可能 |
| **RBAC統合** | 権限管理が自動的にSnowflakeのロール体系に従う |


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()

TICKERS = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'GOOGL', 'META', 'BRK.B',
    'JPM',  'JNJ',  'V',    'PG',   'UNH',  'HD',   'MA',   'MRK',
    'ABBV', 'AVGO', 'KO',   'PEP',  'COST', 'WMT',  'MCD',  'TMO',
    'ABT',  'CRM',  'ACN',  'DHR',  'NKE',  'TXN',  'NEE'
]
ticker_list = "', '".join(TICKERS)

# 株価データ取得
prices_raw = session.sql(f'''
    SELECT TICKER, DATE, VALUE AS ADJ_CLOSE
    FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
    WHERE VARIABLE = 'post-market_close_adjusted'
      AND TICKER IN ('{ticker_list}')
      AND DATE >= '2019-01-01'
    ORDER BY TICKER, DATE
''').to_pandas()

prices_wide = prices_raw.pivot(index='DATE', columns='TICKER', values='ADJ_CLOSE')
prices_wide.index = pd.to_datetime(prices_wide.index)
prices_wide = prices_wide.sort_index().ffill()
print(f"株価データ: {prices_wide.shape[0]} 日 × {prices_wide.shape[1]} 銘柄")


In [ ]:
# SEC ファンダメンタルズ取得（ROE/Revenue/Gross Profit/Leverage）
try:
    sec_raw = session.sql(f'''
        WITH ticker_cik AS (
            SELECT DISTINCT sm.PRIMARY_TICKER AS TICKER, ci.CIK
            FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.COMPANY_INDEX ci
            JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.OPENFIGI_SECURITY_INDEX sm
              ON ARRAY_CONTAINS(ci.COMPANY_ID::VARIANT, sm.COMPANY_IDS)
            WHERE sm.PRIMARY_TICKER IN ('{ticker_list}')
        )
        SELECT tc.TICKER, ra.PERIOD_END_DATE::DATE AS DATE,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION = 'Net income'                THEN TO_NUMERIC(ra.VALUE,38,4) END) AS NET_INCOME,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION = 'Total stockholders equity' THEN TO_NUMERIC(ra.VALUE,38,4) END) AS EQUITY,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION IN ('Revenues','Total revenues','Net revenues') THEN TO_NUMERIC(ra.VALUE,38,4) END) AS REVENUE,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION IN ('Gross profit','Gross Profit') THEN TO_NUMERIC(ra.VALUE,38,4) END) AS GROSS_PROFIT,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION IN ('Long-term debt','Total debt') THEN TO_NUMERIC(ra.VALUE,38,4) END) AS TOTAL_DEBT,
            MAX(CASE WHEN ra.MEASURE_DESCRIPTION IN ('Total assets','Total Assets') THEN TO_NUMERIC(ra.VALUE,38,4) END) AS TOTAL_ASSETS
        FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.SEC_REPORT_ATTRIBUTES ra
        JOIN ticker_cik tc ON ra.CIK = tc.CIK
        WHERE ra.STATEMENT IN ('Income Statement','Balance Sheet')
          AND ra.COVERED_QTRS = 4
          AND ra.PERIOD_END_DATE >= '2019-01-01'
        GROUP BY tc.TICKER, ra.PERIOD_END_DATE
        ORDER BY tc.TICKER, ra.PERIOD_END_DATE
    ''').to_pandas()
    sec_raw['DATE'] = pd.to_datetime(sec_raw['DATE'])
    sec_raw = sec_raw.sort_values(['TICKER','DATE'])
    print(f"SECデータ: {len(sec_raw)} レコード")
except Exception as e:
    print(f"SEC取得失敗({e}) → プロキシを使用")
    sec_raw = None

# センチメントスコア（Block 1B の成果物）
try:
    sent_raw = session.sql('''
        SELECT TICKER, SENTIMENT_JSON:sentiment_score::FLOAT AS SENTIMENT_SCORE
        FROM QUANT_HOL_DB.QUANT.EARNINGS_SENTIMENT
    ''').to_pandas()
    sentiment_map = sent_raw.set_index('TICKER')['SENTIMENT_SCORE'].to_dict()
    print(f"センチメントスコア: {len(sentiment_map)} 銘柄")
except:
    sentiment_map = {}
    print("センチメントスコアなし → 0で補完")


In [ ]:
def cross_sectional_zscore(df):
    """クロスセクショナル Z スコア（各日付で銘柄間標準化）"""
    return df.apply(lambda row: (row - row.mean()) / (row.std() + 1e-8), axis=1)

returns = prices_wide.pct_change()

# Factor 1: MOMENTUM (Jegadeesh & Titman 1993)
ret_12m    = prices_wide.pct_change(252)
ret_1m     = prices_wide.pct_change(21)
momentum   = ret_12m - ret_1m
z_momentum = cross_sectional_zscore(momentum)

# Momentum Trend: 過去6ヶ月でモメンタムが改善しているか
z_momentum_lag63 = z_momentum.shift(63)
momentum_trend   = z_momentum - z_momentum_lag63  # 正 = 改善

# Factor 2: VALUE (Revenue Growth / SEC)
if sec_raw is not None:
    rev_df   = sec_raw.pivot(index='DATE', columns='TICKER', values='REVENUE')
    rev_wide = rev_df.pct_change().resample('D').ffill().reindex(prices_wide.index).ffill()
    z_value  = cross_sectional_zscore(rev_wide)
else:
    z_value  = cross_sectional_zscore(-prices_wide.pct_change(756))

# Factor 3: QUALITY (ROE - Leverage / SEC)
if sec_raw is not None:
    equity_df   = sec_raw.pivot(index='DATE', columns='TICKER', values='EQUITY')
    ni_df       = sec_raw.pivot(index='DATE', columns='TICKER', values='NET_INCOME')
    debt_df     = sec_raw.pivot(index='DATE', columns='TICKER', values='TOTAL_DEBT')
    assets_df   = sec_raw.pivot(index='DATE', columns='TICKER', values='TOTAL_ASSETS')
    roe_df      = (ni_df / equity_df.replace(0, np.nan)).resample('D').ffill().reindex(prices_wide.index).ffill()
    lev_df      = (debt_df / assets_df.replace(0, np.nan)).resample('D').ffill().reindex(prices_wide.index).ffill()
    quality_raw = roe_df - lev_df * 0.3
    z_quality   = cross_sectional_zscore(quality_raw)
else:
    z_quality   = cross_sectional_zscore(1 / (returns.rolling(252).std() + 1e-6))

# Factor 4: LOW VOLATILITY (Frazzini & Pedersen 2014)
z_volatility = cross_sectional_zscore(-returns.rolling(252).std())

# Factor 5: SENTIMENT (Tetlock 2007 / AI_COMPLETE)
valid_cols   = [t for t in TICKERS if t in prices_wide.columns]
sent_series  = pd.Series(sentiment_map).reindex(valid_cols).fillna(5.0)
z_sentiment  = cross_sectional_zscore(
    pd.DataFrame(
        np.tile(sent_series.values, (len(prices_wide), 1)),
        index=prices_wide.index, columns=valid_cols
    )
)

# Composite (Equal-weighted 5 factors, NaN-safe)
composite = (
    z_momentum[valid_cols].fillna(0) +
    z_value[valid_cols].fillna(0)    +
    z_quality[valid_cols].fillna(0)  +
    z_volatility[valid_cols].fillna(0) +
    z_sentiment.reindex(columns=valid_cols).fillna(0)
) / 5

factors = {
    'z_momentum':   z_momentum[valid_cols],
    'z_value':      z_value[valid_cols],
    'z_quality':    z_quality[valid_cols],
    'z_volatility': z_volatility[valid_cols],
    'z_sentiment':  z_sentiment.reindex(columns=valid_cols).fillna(0)
}

print(f"ファクター計算完了: {composite.shape}")
composite.tail(1).T.sort_values(composite.index[-1], ascending=False).head(10)


In [ ]:
# IC (Information Coefficient) 計算と統計的有意性検定
forward_return_21d = prices_wide.pct_change(21).shift(-21)

def compute_monthly_ic(factor_df, return_df):
    """月次 IC を計算し Spearman 相関の有意性を検定"""
    ic_records = []
    freq = 'ME' if pd.__version__ >= '2.2' else 'M'
    factor_monthly = factor_df.resample(freq).last().dropna(how='all')
    return_monthly  = return_df.resample(freq).last().dropna(how='all')

    for d in factor_monthly.index:
        r_candidates = return_monthly.index[return_monthly.index <= d + pd.Timedelta(days=5)]
        if len(r_candidates) == 0:
            continue
        r_date = r_candidates[-1]
        f = factor_monthly.loc[d].dropna()
        r = return_monthly.loc[r_date].reindex(f.index).dropna()
        common = f.index.intersection(r.index)
        if len(common) < 5:
            continue
        ic, pval = stats.spearmanr(f[common], r[common])
        ic_records.append({'DATE': d, 'IC': ic, 'PVAL': pval, 'N': len(common)})
    return pd.DataFrame(ic_records).set_index('DATE')

ic_composite = compute_monthly_ic(composite, forward_return_21d)
ic_momentum  = compute_monthly_ic(factors['z_momentum'][valid_cols], forward_return_21d)
ic_quality   = compute_monthly_ic(factors['z_quality'][valid_cols],  forward_return_21d)
ic_sentiment = compute_monthly_ic(factors['z_sentiment'].reindex(columns=valid_cols).fillna(0), forward_return_21d)

def ic_summary(ic_df, name):
    ic_vals  = ic_df['IC'].dropna()
    mean_ic  = ic_vals.mean()
    std_ic   = ic_vals.std()
    icir     = mean_ic / (std_ic + 1e-8)
    t_stat, p_val = stats.ttest_1samp(ic_vals, 0)
    frac_pos = (ic_vals > 0).mean()
    print(f"\n【{name}】")
    print(f"  Mean IC  : {mean_ic:.4f}  (> 0.05 → 有意)")
    print(f"  ICIR     : {icir:.4f}  (> 0.5  → 実用水準)")
    print(f"  t-stat   : {t_stat:.3f}")
    print(f"  p-value  : {p_val:.4f}  (< 0.05 → 有意)")
    print(f"  % IC > 0 : {frac_pos:.1%}  (> 55% → 安定)")

ic_summary(ic_composite, "Composite")
ic_summary(ic_momentum,  "Momentum")
ic_summary(ic_quality,   "Quality")
ic_summary(ic_sentiment, "Sentiment")


In [ ]:
import matplotlib.pyplot as plt

freq = 'ME' if pd.__version__ >= '2.2' else 'M'
rolling_ic  = ic_composite['IC'].rolling(12).mean()
rolling_mom = ic_momentum['IC'].rolling(12).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(rolling_ic.index,  rolling_ic.values,  label='Composite', lw=2, color='navy')
axes[0].plot(rolling_mom.index, rolling_mom.values, label='Momentum',  lw=1.5, alpha=0.7, color='steelblue')
axes[0].axhline(y=0.05, color='red', ls='--', alpha=0.5, label='IC=0.05 (閾値)')
axes[0].axhline(y=0.0,  color='gray', ls='-', alpha=0.3)
axes[0].set_title('ローリング12ヶ月 IC（ファクター安定性）', fontsize=13)
axes[0].legend(); axes[0].set_ylabel('IC')

axes[1].hist(ic_composite['IC'].dropna(), bins=20, color='navy', alpha=0.7, label='Composite IC')
axes[1].axvline(x=0, color='gray', ls='-')
axes[1].axvline(x=ic_composite['IC'].mean(), color='red', ls='--',
                label=f"Mean={ic_composite['IC'].mean():.4f}")
axes[1].set_title('IC 分布（正規性の確認）', fontsize=13)
axes[1].legend()

plt.tight_layout(); plt.show()


In [ ]:
import lightgbm as lgb
from sklearn.metrics import r2_score

feature_names = ['Z_MOMENTUM','Z_VALUE','Z_QUALITY','Z_VOLATILITY','Z_SENTIMENT','MOMENTUM_TREND']

records = []
z_sent_loop = factors['z_sentiment']

for date in composite.index[252:]:
    for ticker in valid_cols:
        try:
            zm   = factors['z_momentum'].loc[date, ticker]
            zv   = factors['z_value'].loc[date, ticker]
            zq   = factors['z_quality'].loc[date, ticker]
            zvol = factors['z_volatility'].loc[date, ticker]
            zs   = z_sent_loop.loc[date, ticker] if ticker in z_sent_loop.columns else 0.0
            mt   = momentum_trend.loc[date, ticker]
            fwd  = forward_return_21d.loc[date, ticker] if date in forward_return_21d.index else np.nan
            comp = composite.loc[date, ticker]
            if not any(pd.isna([zm, zv, zq, zvol, zs, mt, fwd])):
                records.append({
                    'DATE': date, 'TICKER': ticker,
                    'Z_MOMENTUM': zm, 'Z_VALUE': zv, 'Z_QUALITY': zq,
                    'Z_VOLATILITY': zvol, 'Z_SENTIMENT': zs,
                    'MOMENTUM_TREND': mt, 'COMPOSITE': comp,
                    'FWD_RETURN_21D': fwd
                })
        except:
            pass

factor_df = pd.DataFrame(records)
print(f"学習データ: {len(factor_df):,} サンプル ({factor_df['TICKER'].nunique()} 銘柄)")

split_date = '2023-01-01'
train_df   = factor_df[factor_df['DATE'] < split_date]
test_df    = factor_df[factor_df['DATE'] >= split_date]
X_train, y_train = train_df[feature_names], train_df['FWD_RETURN_21D']
X_test,  y_test  = test_df[feature_names],  test_df['FWD_RETURN_21D']

model = lgb.LGBMRegressor(
    objective='regression', n_estimators=500, learning_rate=0.03,
    num_leaves=31, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
          callbacks=[lgb.early_stopping(50, verbose=False)])

y_pred = model.predict(X_test)
r2     = r2_score(y_test, y_pred)
ic_model, pval_model = stats.spearmanr(y_pred, y_test.values)
t_stat = ic_model / np.sqrt((1 - ic_model**2) / (len(y_test) - 2))

print(f"\n{'='*50}")
print(f"R²           : {r2:.4f}")
print(f"IC (Spearman): {ic_model:.4f}  (目標 > 0.05)")
print(f"p-value      : {pval_model:.4f}  (< 0.05 が有意)")
print(f"t 統計量     : {t_stat:.3f}")
print(f"\n特徴量重要度:")
for name, imp in sorted(zip(feature_names, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:20s}: {imp:.0f}")


In [ ]:
from snowflake.ml.registry import Registry

# バックテスト指標（月次リサンプルで重複リターンバイアスを除去）
test_enriched        = test_df.copy()
test_enriched['PRED'] = y_pred
daily_ls = (
    test_enriched.groupby('DATE')
    .apply(lambda g: g.nlargest(5, 'PRED')['FWD_RETURN_21D'].mean() -
                     g.nsmallest(5, 'PRED')['FWD_RETURN_21D'].mean())
)
freq         = 'ME' if pd.__version__ >= '2.2' else 'M'
monthly_ls   = daily_ls.resample(freq).mean()
sharpe_bt    = monthly_ls.mean() / (monthly_ls.std() + 1e-8) * np.sqrt(12)
ir_bt        = ic_model * np.sqrt(12)
max_dd       = (monthly_ls.cumsum() - monthly_ls.cumsum().cummax()).min()

print(f"Sharpe Ratio     : {sharpe_bt:.3f}")
print(f"Information Ratio: {ir_bt:.3f}  (> 0.5 → 実用水準)")
print(f"最大ドローダウン  : {max_dd:.2%}")

# ML Registry に登録
registry  = Registry(session=session)
model_ref = registry.log_model(
    model=model,
    model_name="FACTOR_RETURN_PREDICTOR",
    version_name="V1",
    sample_input_data=X_train.head(100),
    target_platforms=["WAREHOUSE"],
    comment="5ファクター (Momentum/Value/Quality/Volatility/Sentiment) × LightGBM",
    metrics={
        "r2_test":    round(float(r2), 4),
        "ic":         round(float(ic_model), 4),
        "p_value":    round(float(pval_model), 4),
        "t_stat":     round(float(t_stat), 3),
        "sharpe_bt":  round(float(sharpe_bt), 3),
        "ir_bt":      round(float(ir_bt), 3),
    },
    options={"relax_version": True}
)
print(f"\n✅ ML Registry 登録完了: {model_ref.model_name} / {model_ref.version_name}")


In [ ]:
# FACTOR_SCORES_LATEST テーブルを Snowflake に書き込み
latest_date   = factor_df['DATE'].max()
latest_scores = factor_df[factor_df['DATE'] == latest_date].copy()
latest_scores['PREDICTED_RETURN']  = model.predict(latest_scores[feature_names])
latest_scores['MOMENTUM_IMPROVING'] = (latest_scores['MOMENTUM_TREND'] > 0).astype(int)
latest_scores['MODEL_IC']   = round(float(ic_model), 4)
latest_scores['MODEL_PVAL'] = round(float(pval_model), 4)
latest_scores['MODEL_IR']   = round(float(ir_bt), 3)

cols = ['TICKER','DATE','Z_MOMENTUM','Z_VALUE','Z_QUALITY','Z_VOLATILITY','Z_SENTIMENT',
        'MOMENTUM_TREND','MOMENTUM_IMPROVING','COMPOSITE','PREDICTED_RETURN',
        'MODEL_IC','MODEL_PVAL','MODEL_IR']

session.write_pandas(
    latest_scores[cols],
    table_name='FACTOR_SCORES_LATEST',
    database='QUANT_HOL_DB',
    schema='QUANT',
    overwrite=True
)
print(f"✅ FACTOR_SCORES_LATEST 更新: {len(latest_scores)} 銘柄")
latest_scores.sort_values('COMPOSITE', ascending=False)[
    ['TICKER','Z_MOMENTUM','Z_VALUE','Z_QUALITY','MOMENTUM_IMPROVING',
     'COMPOSITE','PREDICTED_RETURN']
].head(10)


---
## Block 3: ポートフォリオ最適化ツール構築（20分）

### 平均分散最適化（MVO）の限界とCVaR最小化の優位性

**Markowitz (1952) の平均分散最適化（MVO）** はポートフォリオ理論の出発点ですが、  
機関投資家の実務ではいくつかの重大な限界が指摘されています。

| 観点 | MVO（平均分散） | CVaR最小化 |
|------|--------------|----------|
| **リスク指標** | 分散（上下両方の変動） | テールリスク（最悪ケースの平均損失） |
| **最適化問題の性質** | 二次計画（QP） | 線形計画（LP）に変換可能 |
| **外れ値への耐性** | 弱い（分散は外れ値に敏感） | 強い（テール部分のみに注目） |
| **規制対応** | △ | ◎（Basel IIIでVaR→CVaRへ移行） |
| **実装の容易さ** | 分散共分散行列が不安定 | 凸問題として安定的に解ける |

> **実務的な文脈**: 2008年以降、多くの機関投資家はVaR（Value at Risk）からCVaR（Expected Shortfall）へ  
> リスク管理基準を移行しました。CVaRはExtreme Downside Riskをより正確に捉えます。

---

### CVaR（Conditional Value-at-Risk）の数理

**CVaR** は「最悪 $(1-\alpha)$% のシナリオにおける平均損失」です。

$$\text{CVaR}_{\alpha}(w) = \frac{1}{(1-\alpha)T} \sum_{t: r_t^p \leq \text{VaR}_{\alpha}} \left(-r_t^p\right)$$

**直感的な解釈（α = 0.95 の場合）:**
- 過去252日のリターン分布の下位5%（最悪13日間）の平均損失
- この値を最小化することで、テールリスクに強いポートフォリオを構築

```python
# CVaR の計算コア
def cvar_objective(weights):
    port_ret  = ret_matrix @ weights          # 各日のポートフォリオリターン
    var_thresh = np.percentile(port_ret, 5)   # VaR（5パーセンタイル）
    tail       = port_ret[port_ret <= var_thresh]  # テール部分のリターン
    return -tail.mean()                       # CVaR（符号反転で最小化）
```

### Rockafellar & Uryasev (2000) による線形化の重要性

CVaRは元来は非凸問題に見えますが、**補助変数 $\nu$（= VaR）を導入することで線形計画問題に変換できます**。  
これにより：
- グローバル最適解が保証される（局所最適に陥らない）
- `scipy.optimize.minimize` (SLSQP) で高速に解ける
- ポートフォリオサイズが大きくなっても計算時間がスケールしやすい

---

### セクター集中制約の実務的な意義

**なぜウェイト制約（max_weight）だけでは不十分か?**

```
例: NVDA, AAPL, MSFT, GOOGL, META を各25%保有
→ 個別銘柄制約はクリア
→ テクノロジーセクター集中度 = 100%
→ 半導体輸出規制・AI規制など「セクター横断リスク」に無防備
```

本実装では `SECTOR_MAP` でセクターを定義し、**セクター合計ウェイトの上限**を設定します：

```python
# セクター集中制約をSLSQPの制約条件として追加
for sec in unique_sectors:
    idx = [i for i, s in enumerate(sectors) if s == sec]
    constraints.append({
        'type': 'ineq',
        'fun': lambda w, idx=idx: max_sector_weight - np.sum(w[idx])
    })
```

---

### Information Ratio（IR）の定義と実用基準

$$\text{IR} = \frac{E[r_p - r_b]}{\sigma(r_p - r_b)} \cdot \sqrt{252}$$

アクティブリターン（ポートフォリオ - ベンチマーク）の**平均 ÷ 標準偏差**。

| IR 水準 | 解釈 |
|---------|------|
| IR > 1.0 | 卓越したアクティブ運用（上位機関投資家レベル） |
| **IR > 0.5** | **実用水準（機関投資家の採用基準）** |
| IR 0.3〜0.5 | 参考水準 |
| IR < 0.3 | ランダムウォークと有意差なし |

> **本実装の IR 計算**: 等ウェイトポートフォリオをベンチマークとして計算。  
> `IR ≈ IC_monthly × √12` という近似式も広く使われます（Grinold & Kahn, 2000）。

---

### Stored Procedure として実装する理由

ポートフォリオ最適化ロジックを `CREATE PROCEDURE` としてデプロイすることで：

```sql
-- PMがSQLから直接呼び出せる
CALL OPTIMIZE_PORTFOLIO(['AAPL','MSFT',...], 0.25, 0.03, 0.95, 252, 0.40, 0.0);
```

| メリット | 詳細 |
|---------|------|
| **組織共有** | Pythonスクリプトをローカルに持たずともPMが利用可能 |
| **パラメータ追跡** | 実行ごとにパラメータがQuery Historyに記録される |
| **RBAC統合** | 特定ロールにのみ実行権限を付与できる |
| **バージョン管理** | `CREATE OR REPLACE` でロジック更新・git管理 |


In [ ]:
-- OPTIMIZE_PORTFOLIO: CVaR 最小化 + セクター制約 + IR 算出
CREATE OR REPLACE PROCEDURE QUANT_HOL_DB.PM.OPTIMIZE_PORTFOLIO(
    TICKERS           ARRAY,
    MAX_WEIGHT        FLOAT,
    MIN_WEIGHT        FLOAT,
    CVAR_CONFIDENCE   FLOAT,
    LOOKBACK_DAYS     INTEGER,
    MAX_SECTOR_WEIGHT FLOAT,
    TARGET_VOLATILITY FLOAT
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'numpy', 'pandas', 'scipy')
HANDLER = 'optimize_portfolio'
AS $$
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from snowflake.snowpark.context import get_active_session

SECTOR_MAP = {
    'AAPL':'Technology','MSFT':'Technology','NVDA':'Technology','GOOGL':'Technology',
    'META':'Technology','AVGO':'Technology','CRM':'Technology','TXN':'Technology','ACN':'Technology',
    'AMZN':'Consumer Disc','HD':'Consumer Disc','MCD':'Consumer Disc',
    'NKE':'Consumer Disc','COST':'Consumer Disc',
    'JPM':'Financials','MA':'Financials','V':'Financials','BRK.B':'Financials',
    'JNJ':'Healthcare','UNH':'Healthcare','MRK':'Healthcare','ABBV':'Healthcare',
    'TMO':'Healthcare','ABT':'Healthcare','DHR':'Healthcare',
    'PG':'Consumer Staples','KO':'Consumer Staples','PEP':'Consumer Staples','WMT':'Consumer Staples',
    'NEE':'Utilities'
}

def optimize_portfolio(session, tickers, max_weight, min_weight, cvar_confidence,
                       lookback_days, max_sector_weight, target_volatility):
    tickers_list = list(tickers)
    ticker_sql   = "', '".join(tickers_list)
    prices = session.sql(f'''
        SELECT TICKER, DATE, VALUE AS ADJ_CLOSE
        FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
        WHERE VARIABLE = 'post-market_close_adjusted'
          AND TICKER IN ('{ticker_sql}')
          AND DATE >= DATEADD(day, -{lookback_days+10}, CURRENT_DATE())
        ORDER BY TICKER, DATE
    ''').to_pandas()
    if len(prices) == 0:
        return {"error": "No price data found"}
    prices_wide = prices.pivot(index='DATE', columns='TICKER', values='ADJ_CLOSE')
    prices_wide.index = pd.to_datetime(prices_wide.index)
    returns      = prices_wide.pct_change().dropna().iloc[-lookback_days:]
    available    = [t for t in tickers_list if t in returns.columns]
    returns      = returns[available].dropna(axis=1, how='any')
    tickers_used = list(returns.columns)
    n            = len(tickers_used)
    if n < 2:
        return {"error": f"Insufficient data: {tickers_used}"}
    ret_matrix   = returns.values
    def cvar_objective(w):
        pr  = ret_matrix @ w
        var = np.percentile(pr, (1-cvar_confidence)*100)
        tail = pr[pr <= var]
        return -tail.mean() if len(tail) > 0 else 0.0
    constraints = [{'type':'eq','fun': lambda w: np.sum(w)-1}]
    if max_sector_weight < 1.0:
        sectors = [SECTOR_MAP.get(t,'Other') for t in tickers_used]
        for sec in set(sectors):
            idx = [i for i,s in enumerate(sectors) if s==sec]
            if len(idx) > 1:
                constraints.append({'type':'ineq','fun': lambda w,idx=idx: max_sector_weight-np.sum(w[idx])})
    if target_volatility > 0:
        cov = np.cov(ret_matrix.T)
        constraints.append({'type':'ineq','fun': lambda w: target_volatility/np.sqrt(252)-np.sqrt(w@cov@w+1e-12)})
    bounds = [(min_weight, max_weight)]*n
    result = minimize(cvar_objective, np.array([1/n]*n), method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'maxiter':1000,'ftol':1e-9})
    if not result.success:
        return {"error": result.message}
    w       = result.x
    port_r  = ret_matrix @ w
    ann_ret = float(np.mean(port_r)*252)
    ann_vol = float(np.std(port_r)*np.sqrt(252))
    sharpe  = ann_ret/ann_vol if ann_vol>0 else 0.0
    var_95  = float(np.percentile(port_r,5))
    cvar_95 = float(-port_r[port_r<=var_95].mean())
    cum     = np.cumprod(1+port_r)
    max_dd  = float((cum/np.maximum.accumulate(cum)-1).min())
    bench_r = ret_matrix.mean(axis=1)
    active_r= port_r - bench_r
    ir      = float(np.mean(active_r)/(np.std(active_r)+1e-8)*np.sqrt(252))
    sectors = [SECTOR_MAP.get(t,'Other') for t in tickers_used]
    sec_exp = {}
    for t2,ww,s in zip(tickers_used,w,sectors):
        sec_exp[s] = sec_exp.get(s,0)+float(ww)
    allocation = {t:round(float(ww),4) for t,ww in zip(tickers_used,w) if ww>0.001}
    return {
        "status": "success",
        "allocation": dict(sorted(allocation.items(), key=lambda x:-x[1])),
        "sector_exposure": {k:round(v,4) for k,v in sorted(sec_exp.items(),key=lambda x:-x[1])},
        "metrics": {
            "cvar_95_pct":       round(cvar_95*100,3),
            "var_95_pct":        round(-var_95*100,3),
            "ann_return_pct":    round(ann_ret*100,2),
            "ann_volatility_pct":round(ann_vol*100,2),
            "sharpe_ratio":      round(sharpe,3),
            "information_ratio": round(ir,3),
            "max_drawdown_pct":  round(max_dd*100,2)
        },
        "params": {"tickers_used":tickers_used,"lookback_days":lookback_days,
                   "cvar_confidence":cvar_confidence,"max_weight":max_weight,
                   "max_sector_weight":max_sector_weight}
    }
$$;


In [ ]:
-- OPTIMIZE_PORTFOLIO 動作確認
CALL QUANT_HOL_DB.PM.OPTIMIZE_PORTFOLIO(
    ['AAPL','MSFT','NVDA','JPM','JNJ','KO','PG','MA','UNH','COST'],
    0.25,  -- 個別銘柄上限 25%
    0.03,  -- 最小 3%
    0.95,  -- CVaR 95% 信頼水準
    252,   -- 直近1年
    0.40,  -- テクノロジーセクター上限 40%
    0.0    -- ターゲットボラ制約なし
);


In [ ]:
-- GET_FACTOR_SCORES: モメンタムをリアルタイム計算 + ML Registry でリアルタイム推論
-- 従来版との違い:
--   - モメンタムZ: 最新株価から毎回計算（日次で最新）
--   - PREDICTED_RETURN: ML Registry のモデルをリアルタイム呼び出し
--   - Quality/Value/Volatility: 四半期データのため既存テーブルを流用
CREATE OR REPLACE PROCEDURE QUANT_HOL_DB.QUANT.GET_FACTOR_SCORES(
    TOP_N               INTEGER,
    MIN_MOMENTUM_Z      FLOAT,
    MIN_QUALITY_Z       FLOAT,
    MOMENTUM_IMPROVING  BOOLEAN,
    SENTIMENT_FILTER    BOOLEAN,
    MIN_IC_SIGNIFICANCE FLOAT
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pandas', 'numpy', 'snowflake-ml-python')
HANDLER = 'get_factor_scores'
AS $$
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

def cross_sectional_zscore(series):
    return (series - series.mean()) / (series.std() + 1e-8)

def get_factor_scores(session, top_n, min_momentum_z, min_quality_z,
                      momentum_improving, sentiment_filter, min_ic_significance):

    # Step 1: ベーステーブルから銘柄リスト・SEC系ファクター・センチメントを取得
    # Quality/Value/Volatility は四半期ベースのため既存テーブルを流用
    base_df = session.sql('''
        SELECT f.TICKER,
               f.Z_VALUE, f.Z_QUALITY, f.Z_VOLATILITY, f.Z_SENTIMENT,
               f.MODEL_IC, f.MODEL_PVAL, f.MODEL_IR,
               s.SENTIMENT_SCORE, s.GROWTH_OUTLOOK, s.INVESTMENT_SIGNAL
        FROM QUANT_HOL_DB.QUANT.FACTOR_SCORES_LATEST f
        LEFT JOIN (
            SELECT TICKER,
                   SENTIMENT_JSON:sentiment_score::INTEGER AS SENTIMENT_SCORE,
                   SENTIMENT_JSON:growth_outlook::VARCHAR  AS GROWTH_OUTLOOK,
                   SENTIMENT_JSON:investment_signal::VARCHAR AS INVESTMENT_SIGNAL
            FROM QUANT_HOL_DB.QUANT.EARNINGS_SENTIMENT
        ) s ON f.TICKER = s.TICKER
    ''').to_pandas()

    if len(base_df) == 0:
        return {"error": "FACTOR_SCORES_LATEST is empty. Run Block 2 first."}

    tickers     = base_df['TICKER'].tolist()
    ticker_sql  = "', '".join(tickers)

    # Step 2: 最新株価データでモメンタムをリアルタイム計算
    prices_raw = session.sql(f'''
        SELECT TICKER, DATE, VALUE AS ADJ_CLOSE
        FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.STOCK_PRICE_TIMESERIES
        WHERE VARIABLE = 'post-market_close_adjusted'
          AND TICKER IN ('{ticker_sql}')
          AND DATE >= DATEADD(day, -290, CURRENT_DATE())
        ORDER BY TICKER, DATE
    ''').to_pandas()

    prices_wide = prices_raw.pivot(
        index='DATE', columns='TICKER', values='ADJ_CLOSE'
    ).ffill()

    # 12M-1M モメンタム（当日）
    ret_12m = prices_wide.pct_change(252).iloc[-1]
    ret_1m  = prices_wide.pct_change(21).iloc[-1]
    z_momentum_now = cross_sectional_zscore(ret_12m - ret_1m)

    # モメンタムトレンド（63日前のZスコアとの差分）
    if len(prices_wide) > 64:
        ret_12m_lag = prices_wide.pct_change(252).iloc[-64]
        ret_1m_lag  = prices_wide.pct_change(21).iloc[-64]
        z_momentum_lag = cross_sectional_zscore(ret_12m_lag - ret_1m_lag)
        momentum_trend = z_momentum_now - z_momentum_lag
    else:
        momentum_trend = pd.Series(0, index=z_momentum_now.index)

    momentum_improving_flag = (momentum_trend > 0).astype(int)

    # Step 3: 特徴量データフレームを組み立て（ML Registryの入力形式に合わせる）
    result_df = base_df.copy()
    result_df['Z_MOMENTUM']         = result_df['TICKER'].map(z_momentum_now.to_dict()).fillna(0)
    result_df['MOMENTUM_TREND']     = result_df['TICKER'].map(momentum_trend.to_dict()).fillna(0)
    result_df['MOMENTUM_IMPROVING'] = result_df['TICKER'].map(
        momentum_improving_flag.to_dict()
    ).fillna(0).astype(int)

    feature_names = ['Z_MOMENTUM', 'Z_VALUE', 'Z_QUALITY',
                     'Z_VOLATILITY', 'Z_SENTIMENT', 'MOMENTUM_TREND']
    X = result_df[feature_names].fillna(0)

    # Step 4: ML Registry からモデルを取得してリアルタイム推論
    registry  = Registry(session=session)
    model_ref = registry.get_model("FACTOR_RETURN_PREDICTOR").version("V1")
    preds     = model_ref.run(X, function_name="predict")
    result_df['PREDICTED_RETURN'] = (
        preds.values if hasattr(preds, 'values') else list(preds)
    )

    # コンポジットスコアを最新モメンタムで再計算
    result_df['COMPOSITE'] = (
        result_df['Z_MOMENTUM']   + result_df['Z_VALUE'] +
        result_df['Z_QUALITY']    + result_df['Z_VOLATILITY'] +
        result_df['Z_SENTIMENT']
    ) / 5

    # Step 5: フィルタリングとランキング
    mask = (
        (result_df['Z_MOMENTUM'] >= min_momentum_z) &
        (result_df['Z_QUALITY']  >= min_quality_z)
    )
    if momentum_improving:
        mask &= (result_df['MOMENTUM_IMPROVING'] == 1)
    if sentiment_filter:
        mask &= (result_df['SENTIMENT_SCORE'].fillna(0) >= 6)
    if min_ic_significance > 0:
        mask &= (result_df['MODEL_PVAL'].fillna(1) <= min_ic_significance)

    filtered = result_df[mask].sort_values('COMPOSITE', ascending=False).head(top_n)

    return {
        "top_tickers":       filtered['TICKER'].tolist(),
        "scores":            filtered.to_dict(orient='records'),
        "realtime_inference": True,  # ← モデルをリアルタイム呼び出しした証拠
        "model_validity": {
            "ic":     float(filtered['MODEL_IC'].iloc[0])   if len(filtered) > 0 else None,
            "p_value":float(filtered['MODEL_PVAL'].iloc[0]) if len(filtered) > 0 else None,
            "ir":     float(filtered['MODEL_IR'].iloc[0])   if len(filtered) > 0 else None,
        },
    }
$$;


> **リアルタイム推論のポイント**  
> このSPを呼び出すたびに以下が実行されます:
> 1. Marketplaceの最新株価データを取得してモメンタムを計算（日次で最新値）  
> 2. `ML Registry` から `FACTOR_RETURN_PREDICTOR V1` を呼び出して予測  
> 3. 返り値に `"realtime_inference": true` が含まれることを確認  
>
> Snowflake Intelligence（Block 5）からエージェント経由でこのSPが呼ばれるたびに  
> **そのタイミングの最新マーケットデータ × 登録済みモデル** で予測が実行されます。


In [ ]:
-- GET_FACTOR_SCORES 動作確認（SAMデモ水準のスクリーニング）
CALL QUANT_HOL_DB.QUANT.GET_FACTOR_SCORES(
    10,    -- TOP_N
    0.3,   -- MIN_MOMENTUM_Z
    0.0,   -- MIN_QUALITY_Z
    TRUE,  -- MOMENTUM_IMPROVING のみ
    TRUE,  -- SENTIMENT_SCORE >= 6 のみ
    0.10   -- MODEL_PVAL <= 0.10
);


---
## Block 4: Cortex Code × AI エージェント一括構築（25分）

### このブロックの進め方

Cortex Code の **Plan Mode** を使い、自然言語プロンプトだけで以下を自動生成します:

1. **Semantic View × 2** (Cortex Analyst 用 — `quantitative_analyzer` + `financial_analyzer`)
2. **Cortex Search Service** (決算トランスクリプト検索)
3. **Cortex Agent** (6ツール統合エージェント)

> **Cortex Code の起動方法**  
> 右下の ✦ アイコンをクリック → **Plan** スイッチを **ON** にする

### エージェントのツール構成（SAMデモ水準）

| ツール # | 種別 | 役割 | SAMデモの相当物 |
|---------|------|------|--------------|
| ① | Python SP | ファクタースクリーニング | quantitative_analyzer (実行層) |
| ② | Python SP | CVaR ポートフォリオ最適化 | portfolio optimizer |
| ③ | Semantic View | ファクターデータへの NL クエリ | `quantitative_analyzer` |
| ④ | Semantic View | SEC ファンダメンタル検証 | `financial_analyzer` |
| ⑤ | Cortex Search | 決算トランスクリプト検索 | `search_company_events` |
| ⑥ | Cortex Search | 有報 PDF 全文検索 | `search_broker_research` (代替) |

### Step 1: FACTOR_ANALYTICS_VIEW の作成

以下のプロンプトを Cortex Code に貼り付けてください:

```
スキーマ QUANT_HOL_DB.AI を作成してください。
そのスキーマ内に FACTOR_ANALYTICS_VIEW というセマンティックビューを作成してください。

ソーステーブル: QUANT_HOL_DB.QUANT.FACTOR_SCORES_LATEST

ディメンション: TICKER, MOMENTUM_IMPROVING
メジャー: Z_MOMENTUM, Z_VALUE, Z_QUALITY, Z_VOLATILITY, Z_SENTIMENT,
         MOMENTUM_TREND, COMPOSITE, PREDICTED_RETURN, MODEL_IC, MODEL_PVAL, MODEL_IR

サンプル質問:
- "モメンタムスコアが最も高い銘柄トップ5は？"
- "モメンタム改善中かつクオリティがプラスの銘柄は？"
- "コンポジットスコアと予測リターンが両方プラスの銘柄は？"
- "5ファクター全てがプラスの銘柄は？"
```

### Step 2: FUNDAMENTAL_ANALYTICS_VIEW の作成

```
QUANT_HOL_DB.AI スキーマ内に FUNDAMENTAL_ANALYTICS_VIEW というセマンティックビューを作成。

ソーステーブル: SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.SEC_REPORT_ATTRIBUTES
（COMPANY_INDEX と OPENFIGI_SECURITY_INDEX で TICKER に結合）

ディメンション: TICKER, PERIOD_END_DATE, STATEMENT, MEASURE_DESCRIPTION
メジャー: VALUE

主要指標の説明をコメントとして追加:
- 'Net income': 当期純利益
- 'Revenues': 売上高
- 'Gross profit': 売上総利益
- 'Total stockholders equity': 株主資本

サンプル質問:
- "NVIDIAの過去3年の売上成長率を見せて"
- "グロスマージンが最も高い銘柄は？"
- "ROE が最も高い銘柄のランキングを見せて"
- "負債比率が低い銘柄は？"
```


In [ ]:
-- Cortex Code が生成した SQL を確認・実行するセル
-- (Cortex Code が直接このセルを編集します)

-- 確認: FACTOR_ANALYTICS_VIEW
SHOW SEMANTIC VIEWS IN SCHEMA QUANT_HOL_DB.AI;


### Step 3: Cortex Search Service の作成（決算トランスクリプト）

**オプションA（推奨）**: Marketplace の CKE をそのまま使用  
Marketplace で「Cortex Knowledge Extension (Earning Call Transcript)」を取得済みの場合、  
エージェント作成時に identifier を直接指定するだけで RAG が使えます。

**オプションB**: 自前で Cortex Search を構築

以下を Cortex Code に貼り付け:
```
QUANT_HOL_DB.AI スキーマに EARNINGS_TRANSCRIPT_SEARCH という
Cortex Search Serviceを作成してください。

ソーステーブル: SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.EARNINGS_TRANSCRIPTS
検索対象カラム: TRANSCRIPT_TEXT::VARCHAR
返却カラム: TICKERS, EVENT_TIMESTAMP, EVENT_TYPE, TRANSCRIPT_TEXT
ウェアハウス: QUANT_HOL_WH
ターゲットラグ: 1日
```

### Step 4: QUANT_PM_AGENT の作成

以下を Cortex Code に貼り付け:
```
QUANT_HOL_DB.AI スキーマに QUANT_PM_AGENT というCortex Agentを作成してください。

表示名: "クオンツ & ポートフォリオ・アシスタント"

システムプロンプト:
"あなたはクオンツリサーチとポートフォリオマネジメントチームをサポートするAIアシスタントです。

【クオンツアナリストモード】
- 5ファクター（Momentum/Value/Quality/Volatility/Sentiment）によるシステマティックスクリーニング
- モメンタムトレンド改善（過去6ヶ月）の判定
- LightGBMモデルによる21日先リターン予測
- IC/t-stat/IRによる統計的有意性の確認

【ファンダメンタル検証モード】
- SEC提出書類による財務指標検証（売上成長・グロスマージン・ROE・負債比率）
- ファクターシグナルとファンダメンタルズの整合性評価

【PMモード】
- CVaR最小化（95%信頼水準）+ セクター集中制約
- Information Ratio の算出

日本語で回答し、数値は表形式で表示してください。"

サンプル質問:
- "バリュー・クオリティが高くモメンタムが改善中の銘柄をスクリーニングして"
- "モデルの統計的有意性（IC、p-value、IR）を確認して"
- "NVIDIAのSEC提出書類から売上成長とマージン拡大を確認して"
- "スクリーニング銘柄でCVaR最小化・テック集中40%以下のポートフォリオを構築して"

ツール:
1. type=procedure, identifier=QUANT_HOL_DB.QUANT.GET_FACTOR_SCORES
2. type=procedure, identifier=QUANT_HOL_DB.PM.OPTIMIZE_PORTFOLIO
3. type=semantic_view, identifier=QUANT_HOL_DB.AI.FACTOR_ANALYTICS_VIEW
4. type=semantic_view, identifier=QUANT_HOL_DB.AI.FUNDAMENTAL_ANALYTICS_VIEW
5. type=cortex_search (CKEまたはQUANT_HOL_DB.AI.EARNINGS_TRANSCRIPT_SEARCH)
6. type=cortex_search, identifier=QUANT_HOL_DB.QUANT.ANNUAL_REPORT_SEARCH

ウェアハウス: QUANT_HOL_WH
```


---
## Block 4.5: Cortex Agent Skills 登録（10分）

### Agent Skills とは

`SKILL.md` ファイルで定義されたモジュラーな指示セットです。  
エージェントはユーザーのクエリに応じて適切なスキルを自動選択し、  
定義されたワークフローに従ってツールを呼び出します。

Snowflake Intelligence では `/スキル名` でスキルを明示的に呼び出すことができます。

**本リポジトリの `skills/portfolio_optimizer/SKILL.md` を確認してください。**

### PDF アップロード手順

> **★ UI での操作が必要です**
> 1. 左ナビ → **Data** → **Databases** → `QUANT_HOL_DB` → `AI`
> 2. `SKILL_STAGE` をクリック
> 3. `skills/portfolio_optimizer/` フォルダを作成
> 4. `SKILL.md` をアップロード


In [ ]:
-- Skills 用ステージ作成
CREATE STAGE IF NOT EXISTS QUANT_HOL_DB.AI.SKILL_STAGE
    COMMENT = 'Cortex Agent Skills の格納場所';


In [ ]:
-- アップロード確認
LS @QUANT_HOL_DB.AI.SKILL_STAGE/skills/;


In [ ]:
-- エージェントにスキルを追加
ALTER AGENT QUANT_HOL_DB.AI.QUANT_PM_AGENT
    MODIFY LIVE VERSION
    SET SPECIFICATION = $$
    {
        "skills": [
            {
                "name": "portfolio-optimizer",
                "source": {
                    "type": "STAGE",
                    "path": "@QUANT_HOL_DB.AI.SKILL_STAGE/skills/portfolio_optimizer"
                }
            }
        ]
    }
    $$;

-- 確認
DESCRIBE AGENT QUANT_HOL_DB.AI.QUANT_PM_AGENT;


---
## Block 5: Snowflake Intelligence デモ（15分）

### エージェントへのアクセス

左ナビゲーション → **Cortex Agents** → `QUANT_PM_AGENT`  
→ **Preview in Snowflake Intelligence** をクリック

---

### デモ ① クオンツアナリストモード

**ファクタースクリーニング:**
```
モメンタムZスコアが0.5以上かつクオリティがプラスの銘柄をスクリーニングして、
センチメントスコアも一緒に表示してください。
```

**統計的有意性の確認:**
```
現在のモデルのIC、p-value、Information Ratioを確認して、
統計的に有意かどうか評価してください。
```

---

### デモ ② ファンダメンタル検証モード

**SEC データによる検証:**
```
NVIDIAのSEC提出書類から直近の売上成長率とグロスマージンの推移を確認してください。
ファクタースコアと整合しているかも評価してください。
```

---

### デモ ③ PM モード

**ポートフォリオ構築:**
```
コンポジットスコア上位5銘柄を使って、CVaR 95%最小化のポートフォリオを構築してください。
テクノロジーセクターの集中度は40%以下にしてください。
Sharpe RatioとInformation Ratioも表示してください。
```

---

### ★ クライマックス: SAMデモ水準の包括的6ステッププロンプト（全6ツール連携）

```
バリュー、クオリティ、改善するモメンタムに焦点を当てた
マルチファクター株式スクリーニング戦略を構築しています。以下をお願いします：

1. 高いクオリティとバリューのファクターエクスポージャーを持つ銘柄を
   スクリーニング（Z_QUALITY > 0.3 かつ Z_VALUE > 0.0）

2. 過去6ヶ月でモメンタムファクターのトレンドが改善している銘柄
   （MOMENTUM_IMPROVING = 1）を特定

3. SEC提出書類データで財務ファンダメンタルズを検証
   （売上成長・グロスマージン拡大・ROE）

4. 決算説明会の経営陣コメンタリーを確認し、
   クオリティと成長特性を支持する発言を探して

5. 有報・年次報告書で選定銘柄のリスク要因を確認して

6. ファクタースコア・ファンダメンタル検証・センチメント・
   IC/p-valueを含むランク付けリストを作成し、
   CVaR 95%最小化・テック集中40%以下でポートフォリオを構築してください
```

> このプロンプト 1 つで **6 ツールが自動オーケストレーション** されます。  
> SAMデモが示す「戦略開発 3〜5日 → 20分未満」の価値を体験してください。

---

### Skills を使ったプロンプト

Snowflake Intelligence で `/` を入力してスキル一覧を表示:

```
/ → portfolio-optimizer を選択
→「バリュー・クオリティ高くモメンタム改善中の銘柄でCVaR最小化ポートフォリオを
    作って。テック集中40%以下・個別上限25%・Information Ratio も出して」
```
